# SVM (Support Vector Machine)

Resolver el problema de supervivencia del Titanic con Support Vector Machine (SVM) usando el dataset preprocesado.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Carga y limpieza

In [4]:
df = pd.read_csv('dataset.csv')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df_model = pd.get_dummies(df_model, columns=['Sex', 'Embarked'], drop_first=True)

df_model.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,2,0,True,False,True
1,1,1,38.0,1,0,71.2833,2,0,False,False,False
2,1,3,26.0,0,0,7.9250,1,1,False,False,True
3,1,1,35.0,1,0,53.1000,2,0,False,False,True
4,0,3,35.0,0,0,8.0500,1,1,True,False,True


## 2. Separar variables y dividir

In [5]:
X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

# Escalar las características (importante para SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 3. Entrenamiento y evaluacion

In [6]:
model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClasification report:')
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8089887640449438

Clasification report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       110
           1       0.80      0.66      0.73        68

    accuracy                           0.81       178
   macro avg       0.81      0.78      0.79       178
weighted avg       0.81      0.81      0.80       178

Confusion matrix:
[[99 11]
 [23 45]]


## 4. Informacion del modelo

In [7]:
# SVM no tiene feature_importances, mostrar información de vectores de soporte
print(f'Numero de vectores de soporte: {len(model.support_vectors_)}')
print(f'Ratio de vectores de soporte: {len(model.support_vectors_) / len(X_train) * 100:.2f}%')

Numero de vectores de soporte: 328
Ratio de vectores de soporte: 46.13%


## 5. Busqueda simple de hiperparametros

In [8]:
best_score = 0.0
best_kernel = None
best_C = None

for kernel in ['linear', 'rbf', 'poly']:
    for C in [0.1, 1.0, 10.0]:
        temp_model = SVC(kernel=kernel, C=C, gamma='scale', random_state=42)
        temp_model.fit(X_train, y_train)
        score = temp_model.score(X_test, y_test)
        print(f'kernel={kernel}, C={C} -> accuracy={score:.4f}')
        if score > best_score:
            best_score = score
            best_kernel = kernel
            best_C = C

print('\nMejor kernel:', best_kernel)
print('Mejor C:', best_C)
print('Mejor accuracy:', best_score)

kernel=linear, C=0.1 -> accuracy=0.7697
kernel=linear, C=1.0 -> accuracy=0.7697
kernel=linear, C=10.0 -> accuracy=0.7697
kernel=rbf, C=0.1 -> accuracy=0.7921
kernel=rbf, C=1.0 -> accuracy=0.8090
kernel=rbf, C=10.0 -> accuracy=0.7697
kernel=poly, C=0.1 -> accuracy=0.7247
kernel=poly, C=1.0 -> accuracy=0.8258
kernel=poly, C=10.0 -> accuracy=0.7978

Mejor kernel: poly
Mejor C: 1.0
Mejor accuracy: 0.8258426966292135


## 6. Conclusion y explicacion de resultados (con datos reales)

In [9]:
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

print("Conclusion y explicacion (resumen automatico)")
print(f"- Accuracy: {acc:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor kernel: {best_kernel} con C={best_C} y accuracy {best_score:.3f}")
print(f"- Vectores de soporte: {len(model.support_vectors_)} ({len(model.support_vectors_) / len(X_train) * 100:.1f}% del entrenamiento)")

Conclusion y explicacion (resumen automatico)
- Accuracy: 0.809
- Clase 0 (no sobrevive): precision=0.811, recall=0.900
- Clase 1 (sobrevive): precision=0.804, recall=0.662
- Matriz de confusion: TN=99, FP=11, FN=23, TP=45 -> sesgo conservador
- Mejor kernel: poly con C=1.0 y accuracy 0.826
- Vectores de soporte: 328 (46.1% del entrenamiento)
